# Projet Journal Club, groupe 4 : Adam, augmentation par classe, ticket gagnant

Ce notebook réorganise `ProjetJournalClub_ticket_gagnant_4.ipynb` en trois étapes qui s'enchaînent :

| Partie | Contenu | Entraînements |
| --- | --- | --- |
| 1 | Données, CNN du socle, **entraînement avec Adam (lr = 0,005)** sur 3 graines | 3 |
| 2 | **Augmentation de données par classe** (symétries + légère variation de couleur) | 0 (le modèle « Adam + augmentation » est le tour 0 de la partie 3) |
| 3 | **Élagage du ticket gagnant** (Frankle & Carbin) sur les données augmentées, avec Adam | 8 + 2 × 8 + 3 contrôles |
| 4 | Tests (une seule fois) et comparaison des quatre modèles | 0 |

Pourquoi Adam : avec le socle SGD 0,05 + momentum 0,9, le réseau complet **divergeait sur les 3 graines** (exactitude d'entraînement 0,111 = hasard, cf. version 4). Tout le reste du socle est gardé : entropie croisée, décroissance cosinus, batch 128, 100 époques, modèle de la dernière époque.

# 1. Création du CNN

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
import matplotlib.pyplot as plt
!pip install medmnist
from medmnist import PathMNIST,INFO

In [ ]:
class CNN(nn.Module):
    def __init__(self,in_channels, num_classes):
        super(CNN,self).__init__()
        #Bloc 1 : 28x28 -> 14x14, 32 filtres
        self.conv1a = nn.Conv2d(in_channels=in_channels, out_channels=32 ,kernel_size=3, padding=1,stride=1)
        self.conv1b = nn.Conv2d(32,32 ,kernel_size=3, padding=1,stride=1)

        # Bloc 2 14x14 -> 7x7
        self.conv2a = nn.Conv2d(32,64, kernel_size=3,stride=1, padding=1)
        self.conv2b = nn.Conv2d(64,64,kernel_size=3,stride = 1, padding = 1)

        #Bloc 3 7x7 -> 3x3
        self.conv3a = nn.Conv2d(64,128, kernel_size=3,stride=1, padding=1)
        self.conv3b = nn.Conv2d(128,128,kernel_size=3,stride = 1, padding = 1)

        # Max-pool
        self.pool = nn.MaxPool2d(kernel_size=2,stride = 2)

        # Couche dense 128 cartes de tailles 3x3
        self.fc1 = nn.Linear(128*3*3,256)

        # Couche de sortie :
        self.fc2 = nn.Linear(256,num_classes)

    def forward(self,x):
        x = F.relu(self.conv1a(x))
        x = F.relu(self.conv1b(x))
        x = self.pool(x)               # bloc 1

        x = F.relu(self.conv2a(x))
        x = F.relu(self.conv2b(x))
        x = self.pool(x)               # bloc 2

        x = F.relu(self.conv3a(x))
        x = F.relu(self.conv3b(x))
        x = self.pool(x)               # bloc 3

        x = x.reshape(x.shape[0], -1)  # aplatir
        x = F.relu(self.fc1(x))        # couche dense 256
        x = self.fc2(x)
        return x

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
seed_rng = 42
num_classes = 9

### Chargement des données

In [ ]:
def load(split):
    ds = PathMNIST(split=split,download=True,size=28)
    return ds.imgs, ds.labels.reshape(-1)

x_train_full, y_train_full = load("train")
x_val,y_val = load("val")
x_test, y_test = load("test")

### Sous ensemble de 4 500 images

In [ ]:
rng = np.random.default_rng(seed_rng)
idx = np.concatenate([rng.choice(np.where(y_train_full == c)[0], 500, replace=False) for c in range(9)])
x_train, y_train = x_train_full[idx], y_train_full[idx]

### Validation officielle -> 2 000 test du même centre et 8 000 validation restantes

In [ ]:
perm = rng.permutation(len(y_val))
x_test_same, y_test_same = x_val[perm[:2000]], y_val[perm[:2000]]
x_val, y_val = x_val[perm[2000:]], y_val[perm[2000:]]

In [ ]:
print(x_train.shape)

Format des données en N, hauteur, largeur, canaux or nn.Conv2d attend N, canaux, hauteur, largeur

## Analyse préliminaire

In [ ]:
splits = {
    "train officiel": (x_train_full, y_train_full),
    "train 4 500": (x_train, y_train),
    "validation 8 000": (x_val, y_val),
    "test même centre": (x_test_same, y_test_same),
    "test autre centre": (x_test, y_test),
}
names = [INFO["pathmnist"]["label"][str(i)] for i in range (9)]

### Taille et répartition des classes

In [ ]:
# 1. Taille et répartition des classes (en %)
table = pd.DataFrame({k: np.bincount(y, minlength=9) / len(y) * 100 for k, (_, y) in splits.items()},
                     index=names).round(1)
table.loc["effectif"] = [len(y) for _, y in splits.values()]
print(table)

### Exemples d'images par classe

In [ ]:
# 2. Exemples : 6 images par classe
fig, ax = plt.subplots(9, 6, figsize=(6, 10))
for c in range(9):
    imgs = x_train_full[idx][y_train_full[idx] == c][:6]
    for j in range(6):
        ax[c, j].imshow(imgs[j]); ax[c, j].axis("off")
    ax[c, 0].set_title(names[c][:25], fontsize=7, loc="left")
plt.tight_layout()
plt.show()

### Différences entre les centres

In [ ]:
# 3. Décalage entre centres : couleur moyenne par canal
stats = pd.DataFrame({k: np.r_[x.reshape(-1, 3).mean(0), x.reshape(-1, 3).std(0)]
                      for k, (x, _) in splits.items()},
                     index=["moy R", "moy G", "moy B", "std R", "std G", "std B"]).round(1)
print(stats)

fig, ax = plt.subplots(1, 3, figsize=(12, 3))
for i, canal in enumerate("RGB"):
    for k in ["test même centre", "test autre centre"]:
        x = splits[k][0]
        ax[i].hist(x[..., i].reshape(len(x), -1).mean(1), bins=50, alpha=0.5, density=True, label=k)
    ax[i].set_title(f"Intensité moyenne par image, canal {canal}")
ax[0].legend()
plt.tight_layout()
plt.show()

## Données pour l'entraînement

### Centrage et réduction par canal

On utilise x_train afin d'éviter le data leakage

In [ ]:
mean = x_train.reshape(-1, 3).mean(0) / 255.0
std = x_train.reshape(-1, 3).std(0) / 255.0

In [ ]:
def make_dataset(x, y):
    x = torch.tensor(x, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0   # (N, 3, 28, 28)
    x = (x - torch.tensor(mean, dtype=torch.float32).view(1, 3, 1, 1)) \
        / torch.tensor(std, dtype=torch.float32).view(1, 3, 1, 1)
    return TensorDataset(x, torch.tensor(y, dtype=torch.long))

train_set = make_dataset(x_train, y_train)
val_loader = DataLoader(make_dataset(x_val, y_val), batch_size=1024)
test_same_loader = DataLoader(make_dataset(x_test_same, y_test_same), batch_size=1024)
test_other_loader = DataLoader(make_dataset(x_test, y_test), batch_size=1024)

## Evaluation

In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        total_loss += criterion(out, y).item() * len(y)
        correct += (out.argmax(dim=1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n

### Outils d'évaluation complémentaires

- `predict(model, loader)` : vraies classes, classes prédites et probabilités (softmax).
- `scores(model, train_acc)` : sur la validation et les deux tests, l'exactitude, le F1 macro, le rappel de la classe tumorale (`TUM_CLASS = 8`), le F1 de chaque classe et la matrice de confusion, puis les deux écarts (surapprentissage = `train_acc − acc_same`, centre = `acc_same − acc_other`).

Les tests ne sont évalués qu'en partie 4.

In [ ]:
from sklearn.metrics import f1_score, recall_score, confusion_matrix
import copy, time

TUM_CLASS = 8    # "colorectal adenocarcinoma epithelium" : la classe tumorale
print("classe tumorale :", names[TUM_CLASS])

@torch.no_grad()
def predict(model, loader):
    model.eval()
    probs, labels = [], []
    for x, y in loader:
        probs.append(F.softmax(model(x.to(device)), dim=1).cpu())
        labels.append(y)
    probs = np.nan_to_num(torch.cat(probs).numpy(), nan=1 / 9)
    return torch.cat(labels).numpy(), probs.argmax(1), probs

SPLITS = [("val", "validation", val_loader), ("same", "test même centre", test_same_loader),
          ("other", "test autre centre", test_other_loader)]

def scores(model, train_acc):
    out = {}
    for key, _, loader in SPLITS:
        y, p, probs = predict(model, loader)
        out[f"acc_{key}"] = float((y == p).mean())
        out[f"f1_{key}"] = f1_score(y, p, average="macro", zero_division=0)
        out[f"rappel_tum_{key}"] = recall_score(y, p, labels=[TUM_CLASS], average="macro", zero_division=0)
        out[f"f1_classes_{key}"] = f1_score(y, p, labels=list(range(9)), average=None, zero_division=0)
        out[f"cm_{key}"] = confusion_matrix(y, p, labels=list(range(9)), normalize="true")
    out["ecart_surapprentissage"] = train_acc - out["acc_same"]
    out["ecart_centre"] = out["acc_same"] - out["acc_other"]
    return out

## Entraînement : socle commun avec Adam (lr = 0,005)

Une **seule** fonction d'entraînement, `train_model`, sert à toutes les parties :

- `data` : le jeu d'entraînement (`train_set` en partie 1, `train_set_aug` en parties 2 et 3) ;
- `init_state` : des poids de départ (θ0 du ticket, ou une autre initialisation pour le contrôle) ; par défaut, initialisation neuve tirée avec la graine ;
- `masks` : le masque d'élagage ; il est appliqué au départ (`m ⊙ θ0`) et **réappliqué après chaque `optimizer.step()`**. Les poids élagués reçoivent un gradient, et les moments d'Adam les feraient bouger : sans cette ligne, ils « revivraient ».

Réglages : Adam (lr = 0,005, β par défaut, pas de weight decay), entropie croisée, cosinus sur 100 époques, batch 128. La graine fixe l'initialisation **et** l'ordre des batchs (`generator`). Avec la même graine, `train_model(s)` et `train_model(s, init_state=initial_state(s))` donnent exactement le même réseau.

L'exactitude d'entraînement (`train_acc`) est toujours mesurée sur les **4 500 images d'origine**, même quand on entraîne sur les données augmentées : l'écart de surapprentissage reste ainsi comparable d'un modèle à l'autre.

Un entraînement est marqué **divergé** si la perte devient infinie ou si `train_acc` < 0,20 en fin d'entraînement.

In [ ]:
learning_rate = 0.005   # Adam
batch_size = 128
num_epochs = 100
SEEDS = [0, 1, 2]
DIVERGENCE_ACC = 0.20

In [ ]:
def prunable(model):
    # Poids élagables : les poids (pas les biais) des 6 conv et des 2 couches denses
    return {name: p for name, p in model.named_parameters() if name.endswith("weight")}

@torch.no_grad()
def apply_masks(model, masks):
    for name, p in prunable(model).items():
        p.mul_(masks[name])

train_eval_loader = DataLoader(train_set, batch_size=1024)   # 4 500 images d'origine

def train_model(seed, data=None, init_state=None, masks=None,
                epochs=num_epochs, lr=learning_rate, batch_size=batch_size, verbose=False):
    data = train_set if data is None else data
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = CNN(in_channels=3, num_classes=num_classes).to(device)
    if init_state is not None:
        model.load_state_dict(init_state)          # θ0 (ticket) ou autre init (contrôle)
    if masks is not None:
        apply_masks(model, masks)                  # m ⊙ θ0
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    g = torch.Generator().manual_seed(seed)        # même ordre des batchs pour une graine donnée
    train_loader = DataLoader(data, batch_size=batch_size, shuffle=True, generator=g)
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    diverged, epoch = False, 0
    for epoch in range(1, epochs + 1):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            loss = criterion(model(x), y)          # passe avant + perte
            if not torch.isfinite(loss):
                diverged = True
                break
            optimizer.zero_grad()                  # remet les gradients à zéro
            loss.backward()                        # rétropropagation
            optimizer.step()                       # mise à jour des poids
            if masks is not None:
                apply_masks(model, masks)          # les poids élagués restent à 0
        if diverged:
            break
        scheduler.step()                           # décroissance cosinus du taux d'apprentissage

        tr_loss, tr_acc = evaluate(model, train_eval_loader, criterion)
        va_loss, va_acc = evaluate(model, val_loader, criterion)
        for k, v in zip(history, (tr_loss, tr_acc, va_loss, va_acc)):
            history[k].append(v)
        if verbose and epoch % 10 == 0:
            print(f"graine {seed} | époque {epoch:3d} | train {tr_loss:.3f} / {tr_acc:.3f} "
                  f"| val {va_loss:.3f} / {va_acc:.3f}")

    # Modèle de la dernière époque (pas d'early stopping). Les tests ne sont pas évalués ici.
    _, tr_acc = evaluate(model, train_eval_loader, criterion)
    _, va_acc = evaluate(model, val_loader, criterion)
    epoch_min = int(np.argmin(history["val_loss"])) + 1 if history["val_loss"] else epoch
    return model, history, {"train_acc": tr_acc, "val_acc": va_acc, "epoch_min_val_loss": epoch_min,
                            "diverged": diverged or tr_acc < DIVERGENCE_ACC}

def to_cpu(d):
    return {k: v.detach().cpu().clone() for k, v in d.items()}

def model_from_state(state):
    model = CNN(in_channels=3, num_classes=num_classes).to(device)
    model.load_state_dict(state)
    return model

## Résultats sur 3 graines : Adam, sans augmentation

C'est la nouvelle ligne de base du groupe. Les poids finaux sont sauvegardés dans `baseline_adam.pt` pour pouvoir évaluer les tests en partie 4 même après un redémarrage du noyau.

In [ ]:
base_adam = {}
for seed in SEEDS:
    t0 = time.time()
    model, hist, res = train_model(seed, verbose=True)
    base_adam[seed] = {"Pm": 1.0, **res, "history": hist, "state": to_cpu(model.state_dict())}
    print(f"graine {seed} : train {res['train_acc']:.4f} | val {res['val_acc']:.4f} | "
          f"époque min perte val {res['epoch_min_val_loss']} | {time.time() - t0:.0f} s"
          + ("  <-- DIVERGÉ" if res["diverged"] else ""))
torch.save(base_adam, "baseline_adam.pt")

for k in ["train_acc", "val_acc", "epoch_min_val_loss"]:
    v = np.array([base_adam[s][k] for s in SEEDS], dtype=float)
    print(f"{k:24s} {v.mean():.4f} ± {v.std(ddof=1):.4f}")

## Courbes d'entraînement et de validation

In [ ]:
plt.figure(figsize=(6, 4))
for s in SEEDS:
    h = base_adam[s]["history"]
    line, = plt.plot(h["train_loss"], label=f"train (graine {s})")
    plt.plot(h["val_loss"], "--", color=line.get_color(), label=f"val (graine {s})")
plt.xlabel("époque"); plt.ylabel("perte"); plt.title("Adam lr = 0,005, sans augmentation")
plt.legend(); plt.tight_layout(); plt.show()

# 2. Augmentation de données par classe

Le sous-ensemble d'entraînement est équilibré (500 images par classe). On ajoute, **classe par classe**, des copies transformées des images de cette classe. Le nombre d'images ajoutées est réglé par classe dans `AUG_PAR_CLASSE` : par défaut 500 pour chaque classe (on double chaque classe, le jeu reste équilibré : 9 000 images). Pour renforcer une classe faible (F1 bas dans la partie 4), il suffit d'augmenter sa valeur.

**Transformations** (choisies pour l'histologie) :

1. une des **7 symétries non triviales du carré** (rotations de 90°, 180°, 270°, retournements) : un tissu n'a pas d'orientation, la classe ne change pas ;
2. une **légère variation de couleur** par canal (contraste et luminosité dans ±10 %), pour imiter les différences de coloration entre centres vues dans l'analyse préliminaire (le test « autre centre » a des écarts-types de couleur plus grands).

Règles pour éviter les fuites :

- l'augmentation ne touche que l'entraînement ; validation et tests sont inchangés ;
- le centrage-réduction garde les moyennes et écarts-types des 4 500 images d'origine (`mean`, `std`) ;
- dans une classe, chaque image source est utilisée au plus une fois tant que `AUG_PAR_CLASSE[c]` ≤ 500 ;
- l'augmentation est tirée **une fois** avec `AUG_SEED` : toutes les graines et tous les tours d'élagage voient exactement les mêmes images.

In [ ]:
AUG_SEED = 2026
AUG_PAR_CLASSE = {c: 500 for c in range(num_classes)}   # images ajoutées par classe
COLOR_JITTER = 0.10                                      # ±10 % de contraste et de luminosité

def augment_image(img, rng):
    t = rng.integers(1, 8)                               # 1..7 : jamais l'identité
    img = np.rot90(img, k=t % 4, axes=(0, 1))
    if t >= 4:
        img = img[:, ::-1]
    img = img.astype(np.float32)
    m = img.mean(axis=(0, 1), keepdims=True)
    contraste = rng.uniform(1 - COLOR_JITTER, 1 + COLOR_JITTER, size=(1, 1, 3))
    luminosite = rng.uniform(1 - COLOR_JITTER, 1 + COLOR_JITTER, size=(1, 1, 3))
    img = ((img - m) * contraste + m) * luminosite
    return np.clip(img, 0, 255).astype(np.uint8)

def augment_par_classe(x, y, n_par_classe, seed):
    rng = np.random.default_rng(seed)
    xs, ys, src = [x], [y], []
    for c, n in n_par_classe.items():
        if n <= 0:
            continue
        idx_c = np.where(y == c)[0]
        pick = rng.choice(idx_c, n, replace=n > len(idx_c))
        xs.append(np.stack([augment_image(x[i], rng) for i in pick]))
        ys.append(np.full(n, c, dtype=y.dtype))
        src.append(pick)
    return np.concatenate(xs), np.concatenate(ys), np.concatenate(src)

x_train_aug, y_train_aug, aug_src = augment_par_classe(x_train, y_train, AUG_PAR_CLASSE, AUG_SEED)
train_set_aug = make_dataset(x_train_aug, y_train_aug)

print(pd.DataFrame({"avant": np.bincount(y_train, minlength=9),
                    "après": np.bincount(y_train_aug, minlength=9)}, index=names))
print("total :", len(y_train), "->", len(y_train_aug))

### Vérification visuelle : une image d'origine et trois versions augmentées par classe

In [ ]:
rng_vis = np.random.default_rng(0)
fig, ax = plt.subplots(9, 4, figsize=(4.5, 10))
for c in range(9):
    img = x_train[y_train == c][0]
    ax[c, 0].imshow(img)
    for j in range(1, 4):
        ax[c, j].imshow(augment_image(img, rng_vis))
    for a in ax[c]:
        a.axis("off")
    ax[c, 0].set_title(names[c][:25], fontsize=7, loc="left")
ax[0, 0].text(0, -12, "origine", fontsize=7); ax[0, 1].text(0, -12, "augmentées →", fontsize=7)
plt.tight_layout(); plt.show()

# Couleur moyenne : l'augmentation élargit un peu la dispersion, la moyenne ne bouge pas
print(pd.DataFrame({k: np.r_[x.reshape(-1, 3).mean(0), x.reshape(-1, 3).std(0)]
                    for k, x in [("train 4 500", x_train), ("train augmenté", x_train_aug),
                                 ("test autre centre", x_test)]},
                   index=["moy R", "moy G", "moy B", "std R", "std G", "std B"]).round(1))

# 3. Élagage du ticket gagnant (Frankle & Carbin, 2019), Adam + augmentation

**Principe** (élagage itératif, « stratégie 1 » de l'annexe B) :

1. On tire une initialisation θ0 du CNN et on la sauvegarde.
2. On entraîne le réseau (Adam 0,005, données augmentées, 100 époques).
3. On élague, couche par couche, les poids de plus petite valeur absolue → masque `m`.
4. On remet les poids survivants à leur valeur dans θ0 : `m ⊙ θ0` est le ticket.
5. On recommence 2 → 4.

**Fixé** (Conv-6 du papier) : élagage non structuré par magnitude, couche par couche ; par tour, 15 % des poids restants dans les conv, 20 % dans `fc1`, 10 % dans `fc2` ; retour à θ0 à chaque tour ; chaque tour est un entraînement complet identique.
**Réglé** : le nombre de tours n, donc la fraction de poids restants Pm. Grille `GRID_ROUNDS`.

Le **tour 0** d'une trajectoire est le réseau complet entraîné avec Adam sur les données augmentées : c'est le modèle **« Adam + augmentation »** de la partie 2 (même graine, même initialisation, même ordre des batchs). Il n'est donc pas entraîné à part.

Tout se fait sur les **données augmentées** (`TRAIN_DATA = train_set_aug`).

## 3.1 Réglages du protocole

- `PRUNE_RATE` : fraction des poids **encore vivants** retirée à chaque tour. Après n tours, il reste 0,85ⁿ des poids conv, 0,80ⁿ de `fc1` et 0,90ⁿ de `fc2`.
- `TOL_SELECTION` : règle de choix fixée **avant** de voir les résultats. Meilleure exactitude de validation ; si plusieurs valeurs sont à moins de 0,5 point, on prend la plus élaguée.
- `REINIT_OFFSET` : le contrôle réinitialisé de la graine s utilise l'initialisation de la graine s + 1000.

In [ ]:
TRAIN_DATA = train_set_aug

# Taux d'élagage PAR TOUR (Frankle & Carbin, Conv-6 ; sortie = moitié du taux dense)
PRUNE_RATE = {"conv": 0.15, "fc1": 0.20, "fc2": 0.10}

GRID_ROUNDS = [2, 3, 4, 5, 7]          # grille du nombre de tours n (=> Pm)
N_ROUNDS_TUNE = max(GRID_ROUNDS)
SEED_TUNE = 0                          # 1 graine pour le réglage
REINIT_OFFSET = 1000                   # init du contrôle = graine + 1000
TOL_SELECTION = 0.005                  # 0,5 point d'exactitude de validation

## 3.2 Fonctions d'élagage

On ne supprime pas physiquement les poids : chaque tenseur de poids `W` a un masque `M` (0/1) de même forme, et le réseau utilise `W * M`. `prunable` et `apply_masks` sont définies avec `train_model`.

- `layer_rate(name)` : taux par tour d'une couche (`conv…` → 0,15 ; `fc1` → 0,20 ; `fc2` → 0,10).
- `full_masks(model)` : masques remplis de 1 (tour 0).
- `prune_step(model, masks)` : **un tour d'élagage**. Pour chaque couche, `k = taux × nombre de vivants`, puis on met à 0 les `k` vivants de plus petite valeur absolue. Les poids déjà morts reçoivent un score infini pour ne jamais être re-choisis.
- `sparsity(masks)` : Pm global, nombre de poids gardés et pourcentage restant par couche.

In [ ]:
def layer_rate(name):
    if name.startswith("conv"):
        return PRUNE_RATE["conv"]
    return PRUNE_RATE[name.split(".")[0]]          # "fc1" ou "fc2"

def full_masks(model):
    return {name: torch.ones_like(p) for name, p in prunable(model).items()}

@torch.no_grad()
def prune_step(model, masks):
    new_masks = {}
    for name, p in prunable(model).items():
        m = masks[name]
        alive = m.bool()
        if not torch.isfinite(p[alive]).all():
            raise ValueError(f"{name} : poids NaN/infinis, on doit élaguer le modèle sain de l'étape d'avant")
        k = int(round(layer_rate(name) * int(alive.sum())))
        new_m = m.clone()
        if k > 0:
            scores_abs = p.abs().masked_fill(~alive, float("inf"))   # poids déjà morts : jamais re-choisis
            idx = torch.topk(scores_abs.flatten(), k, largest=False).indices
            new_m.view(-1)[idx] = 0
        new_masks[name] = new_m
    return new_masks

def sparsity(masks):
    kept = sum(int(m.sum()) for m in masks.values())
    total = sum(m.numel() for m in masks.values())
    per_layer = {n: round(100 * float(m.mean()), 1) for n, m in masks.items()}
    return kept / total, kept, per_layer

def initial_state(seed):
    # θ0 : exactement l'initialisation de train_model(seed)
    torch.manual_seed(seed)
    model = CNN(in_channels=3, num_classes=num_classes).to(device)
    return copy.deepcopy(model.state_dict())

## 3.3 Trajectoire d'élagage itératif

```
θ0 = initial_state(seed) ; masque = 1 partout
pour r = 0 … n_rounds :
    entraîner m ⊙ θ0 sur TRAIN_DATA          (train_model avec init_state et masks)
    enregistrer Pm, exactitudes, poids finaux, masque
    si r < n_rounds : masque = prune_step(poids entraînés, masque)
```

Le retour à θ0 est implicite : chaque tour repart de `theta0`, seul le masque change. Si un tour diverge, le tour suivant élague à partir du dernier modèle sain.

In [ ]:
def lottery_trajectory(seed, n_rounds=None, target_Pm=None):
    assert n_rounds is not None or target_Pm is not None
    theta0 = initial_state(seed)
    masks = full_masks(model_from_state(theta0))
    last_ok_model = model_from_state(theta0)   # si le tour 0 diverge : élagage sur θ0
    rounds, r = [], 0
    while True:
        t0 = time.time()
        model, hist, res = train_model(seed, data=TRAIN_DATA, init_state=theta0, masks=masks)
        Pm, kept, per_layer = sparsity(masks)
        rounds.append({"round": r, "Pm": Pm, "n_poids": kept, "per_layer": per_layer, **res,
                       "history": hist, "state": to_cpu(model.state_dict()), "masks": to_cpu(masks)})
        print(f"graine {seed} | tour {r:2d} | Pm {100 * Pm:5.1f} % | train {res['train_acc']:.3f} "
              f"| val {res['val_acc']:.3f} | {time.time() - t0:.0f} s" + ("  <-- DIVERGÉ" if res["diverged"] else ""))
        if (n_rounds is not None and r >= n_rounds) or (target_Pm is not None and Pm <= target_Pm + 1e-9):
            return rounds
        if not res["diverged"]:
            last_ok_model = model
        masks = prune_step(last_ok_model, masks)
        r += 1

## 3.4 Réglage de Pm (graine 0, validation uniquement)

Trajectoire de `N_ROUNDS_TUNE` tours avec la graine 0, lecture des points de la grille, choix de n\* par la règle fixée à l'avance. Le tour 0 (Adam + augmentation, réseau complet) est affiché comme repère. Sauvegarde dans `ticket_reglage.pt`.

In [ ]:
traj_tune = lottery_trajectory(SEED_TUNE, N_ROUNDS_TUNE)
torch.save(traj_tune, "ticket_reglage.pt")

grid = pd.DataFrame([{"tours n": r["round"], "Pm (%)": round(100 * r["Pm"], 1),
                      "train_acc": r["train_acc"], "val_acc": r["val_acc"],
                      "écart train − val": r["train_acc"] - r["val_acc"],
                      "époque min perte val": r["epoch_min_val_loss"], "divergé": r["diverged"]}
                     for r in traj_tune if r["round"] in [0] + GRID_ROUNDS])
print(grid.round(4).to_string(index=False))

cand = grid[(grid["tours n"] > 0) & (~grid["divergé"])]
best = cand["val_acc"].max()
n_star = int(cand.loc[cand["val_acc"] >= best - TOL_SELECTION, "tours n"].max())
Pm_star = traj_tune[n_star]["Pm"]
print(f"\nValeur retenue : n* = {n_star} tours, Pm* = {100 * Pm_star:.1f} %")

## 3.5 Graines 1 et 2 + contrôle réinitialisé

1. **Tickets** : la graine 0 est déjà calculée ; on calcule les trajectoires des graines 1 et 2 jusqu'à Pm\*. Chaque graine a son propre θ0 et donc son propre masque.
2. **Contrôle** (« random reinitialization ») : même masque que le ticket au tour n\*, mais nouvelle initialisation (graine + 1000), même ordre des batchs. Si le ticket fait mieux, c'est θ0 qui compte : vrai ticket gagnant.

In [ ]:
final = {SEED_TUNE: traj_tune[: n_star + 1]}
for s in SEEDS:
    if s not in final:
        final[s] = lottery_trajectory(s, target_Pm=Pm_star)

controls = {}
for s in SEEDS:
    masks_star = {k: v.to(device) for k, v in final[s][n_star]["masks"].items()}
    theta_reinit = initial_state(s + REINIT_OFFSET)
    model, hist, res = train_model(s, data=TRAIN_DATA, init_state=theta_reinit, masks=masks_star)
    Pm, kept, _ = sparsity(masks_star)
    controls[s] = {"Pm": Pm, "n_poids": kept, **res, "history": hist, "state": to_cpu(model.state_dict())}
    print(f"contrôle graine {s} | Pm {100 * Pm:.1f} % | train {res['train_acc']:.3f} | val {res['val_acc']:.3f}"
          + ("  <-- DIVERGÉ" if res["diverged"] else ""))

torch.save({"final": final, "controls": controls, "n_star": n_star}, "ticket_final.pt")

# 4. Tests (utilisés une seule fois) et comparaison

Pour chaque graine, quatre modèles :

| Modèle | Données | Poids | Rôle |
| --- | --- | --- | --- |
| `Adam` | 4 500 d'origine | 100 % | ligne de base (partie 1) |
| `Adam + aug.` | augmentées | 100 % | effet de l'augmentation (tour 0) |
| `ticket` | augmentées | Pm\* | effet de l'élagage ticket gagnant |
| `contrôle réinit.` | augmentées | Pm\* | même masque, autre initialisation |

**Comparaisons appariées** (même graine, donc même θ0 et même ordre des batchs ; seule la différence étudiée change) :

- `Adam + aug. − Adam` : ce qu'apporte l'augmentation ;
- `ticket − Adam + aug.` : ce qu'apporte l'élagage, à données égales ;
- `ticket − contrôle réinit.` : l'initialisation θ0 compte-t-elle ?
- `ticket − Adam` : effet total des deux régularisations.

Règle de l'énoncé : un écart moyen plus petit que son écart-type sur les 3 graines n'est pas conclusif (colonne `conclusif`).

Si le noyau a redémarré, la cellule suivante recharge les fichiers `.pt` (les cellules des parties 1 et 2 qui définissent les données et les fonctions doivent avoir été relancées, sans les entraînements).

In [ ]:
if "final" not in globals():
    base_adam = torch.load("baseline_adam.pt", weights_only=False)
    traj_tune = torch.load("ticket_reglage.pt", weights_only=False)
    saved = torch.load("ticket_final.pt", weights_only=False)
    final, controls, n_star = saved["final"], saved["controls"], saved["n_star"]
    Pm_star = traj_tune[n_star]["Pm"]

METRICS = ["acc_val", "acc_same", "acc_other", "f1_val", "f1_same", "f1_other",
           "rappel_tum_same", "rappel_tum_other", "ecart_surapprentissage", "ecart_centre"]
MODELS = ["Adam", "Adam + aug.", "ticket", "contrôle réinit."]

def score_rows(entries):
    # entries : liste de (graine, modèle, enregistrement avec state / train_acc / diverged / Pm)
    return [{"graine": s, "modèle": lab, "Pm (%)": round(100 * rec["Pm"], 1), "train_acc": rec["train_acc"],
             "divergé": bool(rec["diverged"]), "epoch_min_val_loss": rec["epoch_min_val_loss"],
             **scores(model_from_state(rec["state"]), rec["train_acc"])} for s, lab, rec in entries]

def paired_deltas(df, a, b, seeds):
    A = df[df["modèle"] == a].set_index("graine").loc[seeds, METRICS]
    B = df[df["modèle"] == b].set_index("graine").loc[seeds, METRICS]
    d = A - B
    out = pd.DataFrame({"écart moyen": d.mean(), "écart-type": d.std(ddof=1) if len(seeds) > 1 else np.nan})
    out["conclusif"] = np.where(out["écart moyen"].abs() > out["écart-type"], "oui", "non")
    return out

entries = []
for s in SEEDS:
    entries += [(s, "Adam", base_adam[s]), (s, "Adam + aug.", final[s][0]),
                (s, "ticket", final[s][n_star]), (s, "contrôle réinit.", controls[s])]
res_df = pd.DataFrame(score_rows(entries))
res_df.drop(columns=[c for c in res_df if c.startswith(("f1_classes", "cm_"))]).to_csv(
    "resultats_adam_aug_ticket.csv", index=False)

show = ["Pm (%)", "train_acc", "divergé"] + METRICS
print(f"Ticket : n* = {n_star} tours, Pm* = {100 * Pm_star:.1f} %\n")
print("=== Par graine ===")
print(res_df[["graine", "modèle"] + show].round(4).to_string(index=False))

summ = res_df.groupby("modèle", sort=False)[METRICS + ["epoch_min_val_loss"]].agg(["mean", lambda v: v.std(ddof=1)])
summ.columns = [f"{c} {'moy' if f == 'mean' else 'std'}" for c, f in summ.columns]
print("\n=== Moyenne ± écart-type sur 3 graines ===")
print(summ.T.round(4).to_string())

for a, b in [("Adam + aug.", "Adam"), ("ticket", "Adam + aug."), ("ticket", "contrôle réinit."), ("ticket", "Adam")]:
    print(f"\n=== Écarts appariés : {a} − {b} ===")
    print(paired_deltas(res_df, a, b, SEEDS).round(4).to_string())

## 4.1 Figures

1. **Colline d'Occam** : exactitude de validation et écart entraînement − validation selon Pm (échelle log, axe inversé). Pointillés : tour 0 (Adam + aug.) de chaque graine.
2. **Courbes de perte**, graine 0 : Adam, Adam + aug., ticket, contrôle.
3. **Exactitude, F1 macro, rappel tumoral** sur validation et tests (moyenne ± écart-type).
4. **F1 par classe** (tests même centre et autre centre) : quelles classes profitent de l'augmentation et de l'élagage.
5. **Matrices de confusion moyennes**, test autre centre.

In [ ]:
def _mean_sd(values):
    v = np.asarray(values, dtype=float)
    return v.mean(), (v.std(ddof=1) if len(v) > 1 else 0.0)

def plot_compare(df, models, title):
    fig, ax = plt.subplots(1, 3, figsize=(17, 4))
    for a, (m, mlab) in zip(ax, [("acc", "Exactitude"), ("f1", "F1 macro"), ("rappel_tum", "Rappel tumeur")]):
        x, w = np.arange(len(models)), 0.27
        for i, (key, klab, _) in enumerate(SPLITS):
            col = f"{m}_{key}"
            if col not in df:
                continue
            ms = [_mean_sd(df.loc[df["modèle"] == mo, col]) for mo in models]
            a.bar(x + (i - 1) * w, [v[0] for v in ms], w, yerr=[v[1] for v in ms], capsize=4, label=klab)
        a.set_xticks(x); a.set_xticklabels(models, fontsize=8); a.set_ylim(0, 1); a.set_title(mlab)
    ax[0].legend(fontsize=8); fig.suptitle(title); plt.tight_layout(); plt.show()

def plot_f1_classes(df, models, key="other", title=""):
    x, w = np.arange(9), 0.8 / len(models)
    plt.figure(figsize=(13, 4))
    for i, mo in enumerate(models):
        arr = np.stack(df.loc[df["modèle"] == mo, f"f1_classes_{key}"].values)
        sd = arr.std(0, ddof=1) if len(arr) > 1 else None
        plt.bar(x + (i - (len(models) - 1) / 2) * w, arr.mean(0), w, yerr=sd, capsize=3, label=mo)
    plt.xticks(x, [n[:22] for n in names], rotation=30, ha="right", fontsize=8)
    plt.ylabel("F1"); plt.ylim(0, 1); plt.title(title); plt.legend(fontsize=8); plt.tight_layout(); plt.show()

def plot_cm(cm, title, ax):
    ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    for i in range(9):
        for j in range(9):
            ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if cm[i, j] > 0.5 else "black")
    ax.set_xticks(range(9)); ax.set_yticks(range(9))
    ax.set_yticklabels([n[:18] for n in names], fontsize=7)
    ax.set_xlabel("classe prédite"); ax.set_ylabel("vraie classe"); ax.set_title(title, fontsize=9)

def plot_cms(df, models, key="other", title=""):
    fig, ax = plt.subplots(1, len(models), figsize=(6 * len(models), 5.5))
    for a, mo in zip(np.atleast_1d(ax), models):
        cms = df.loc[df["modèle"] == mo, f"cm_{key}"].values
        plot_cm(np.mean(np.stack(cms), axis=0), f"{mo} ({len(cms)} graine(s))", a)
    fig.suptitle(title); plt.tight_layout(); plt.show()

# 1. Colline d'Occam
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for s in SEEDS:
    traj = traj_tune if s == SEED_TUNE else final[s]
    pm = [100 * r["Pm"] for r in traj]
    ax[0].plot(pm, [r["val_acc"] for r in traj], "o-", label=f"graine {s}")
    ax[1].plot(pm, [r["train_acc"] - r["val_acc"] for r in traj], "o-", label=f"graine {s}")
    ax[0].axhline(traj[0]["val_acc"], ls=":", color=ax[0].lines[-1].get_color())
for a, t in zip(ax, ["Exactitude de validation", "Écart entraînement − validation"]):
    a.set_xscale("log"); a.invert_xaxis(); a.set_xlabel("Poids restants Pm (%)"); a.set_title(t)
    a.axvline(100 * Pm_star, color="grey", ls="--")
ax[0].legend(); plt.tight_layout(); plt.show()

# 2. Courbes de perte, graine 0
plt.figure(figsize=(7, 4.5))
for label, h in [("Adam", base_adam[SEED_TUNE]["history"]), ("Adam + aug.", traj_tune[0]["history"]),
                 ("ticket", traj_tune[n_star]["history"]), ("contrôle réinit.", controls[SEED_TUNE]["history"])]:
    line, = plt.plot(h["train_loss"], label=f"{label} (train)")
    plt.plot(h["val_loss"], "--", color=line.get_color(), label=f"{label} (val)")
plt.xlabel("époque"); plt.ylabel("perte"); plt.title(f"Graine {SEED_TUNE}"); plt.legend(fontsize=7)
plt.tight_layout(); plt.show()

# 3 à 5
plot_compare(res_df, MODELS, f"Adam, augmentation et ticket (Pm* = {100 * Pm_star:.1f} %), 3 graines")
plot_f1_classes(res_df, MODELS, "other", "F1 par classe, test autre centre (moyenne des graines)")
plot_f1_classes(res_df, MODELS, "same", "F1 par classe, test même centre (moyenne des graines)")
plot_cms(res_df, ["Adam", "Adam + aug.", "ticket"], "other", "Matrices de confusion moyennes, test autre centre")

## 4.2 Évolution de la généralisation selon Pm

Pour chaque tour de chaque graine, mêmes scores sur la validation et les deux tests ; moyenne ± écart-type selon Pm. Croix rouges : tours où au moins une graine a divergé. Pointillés : `Adam` (sans augmentation, partie 1), moyenne des 3 graines.

**Attention :** ces courbes de test sont **descriptives**. Le choix de Pm\* est fait sur la validation seulement (3.4).

In [ ]:
def trajectory_rows(traj, seed, serie):
    rows = []
    for r in traj:
        sc = scores(model_from_state(r["state"]), r["train_acc"])
        rows.append({"série": serie, "graine": seed, "tour": r["round"], "Pm (%)": round(100 * r["Pm"], 1),
                     "train_acc": r["train_acc"], "divergé": bool(r["diverged"]),
                     **{k: v for k, v in sc.items() if not k.startswith(("f1_classes", "cm_"))}})
    return rows

def plot_evolution(ev, refs=None, title=""):
    fig, ax = plt.subplots(2, 3, figsize=(18, 8))
    for row, (m, mlab) in enumerate([("acc", "Exactitude"), ("f1", "F1 macro")]):
        for col, (key, klab, _) in enumerate(SPLITS):
            a, c = ax[row, col], f"{m}_{key}"
            for serie, d in ev.groupby("série", sort=False):
                g = d.groupby("Pm (%)")[c]
                pm = np.array(sorted(g.groups, reverse=True))
                line = a.errorbar(pm, g.mean().reindex(pm).values, yerr=np.nan_to_num(g.std(ddof=1).reindex(pm).values),
                                  fmt="o-", capsize=3, label=serie)
                pm_div = np.unique(d.loc[d["divergé"], "Pm (%)"])
                if len(pm_div):
                    a.scatter(pm_div, g.mean().reindex(pm_div).values, marker="x", s=90, color="red", zorder=5)
                if refs is not None:
                    a.axhline(refs[c], ls=":", color="black", label="Adam sans aug." if (row, col) == (0, 0) else None)
            a.set_xscale("log"); a.invert_xaxis(); a.set_title(f"{mlab} — {klab}")
            a.set_xlabel("Poids restants Pm (%)")
    ax[0, 0].legend(fontsize=8); fig.suptitle(title); plt.tight_layout(); plt.show()

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    for a, (c, t) in zip(ax, [("ecart_surapprentissage", "Écart train − test même centre (surapprentissage)"),
                              ("ecart_centre", "Écart même centre − autre centre")]):
        for serie, d in ev.groupby("série", sort=False):
            g = d.groupby("Pm (%)")[c]
            pm = np.array(sorted(g.groups, reverse=True))
            a.errorbar(pm, g.mean().reindex(pm).values, yerr=np.nan_to_num(g.std(ddof=1).reindex(pm).values),
                       fmt="o-", capsize=3, label=serie)
            if refs is not None:
                a.axhline(refs[c], ls=":", color="black")
        a.set_xscale("log"); a.invert_xaxis(); a.set_xlabel("Poids restants Pm (%)"); a.set_title(t)
    ax[0].legend(fontsize=8); plt.tight_layout(); plt.show()

ev = pd.DataFrame([row for s in SEEDS
                   for row in trajectory_rows(traj_tune if s == SEED_TUNE else final[s], s, "Adam + aug. (3 graines)")])
ref_adam = res_df[res_df["modèle"] == "Adam"][METRICS].mean()

print("=== Scores moyens sur les 3 graines, par tour ===")
print(ev.groupby(["tour", "Pm (%)"])[METRICS].mean().round(4).to_string())
plot_evolution(ev, ref_adam, "Ticket gagnant, Adam + augmentation : généralisation selon Pm "
                             "(pointillés = Adam sans augmentation)")